1. Ingest ----> 2. prepare ----->  3. store -----> 4. retreivement -------> 5. augment -------> . generation

In [ ]:
# step 0 : Importing the tools 
from groq import Groq
import os
from dotenv import load_dotenv
load_dotenv()

api_key=os.getenv("api_key")

client=Groq(api_key=api_key)

In [ ]:
# for loading the documents
from langchain_community.document_loaders import PyPDFDirectoryLoader
# for chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
# encoding of the chunks
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
# for storing the chunks 
from langchain_community.vectorstores import Chroma

## prepare
### Chunking

In [ ]:
# Loading the pdf
pdfs_loader=PyPDFDirectoryLoader("tesla-annual-reports")
#Chunking
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=512,
    chunk_overlap=16
)

tesla_chunks=pdfs_loader.load_and_split(text_splitter)


## Store

In [ ]:
tesla_collection="tesla-10k-2019-to-2023"

embedding_model=SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
# embedding and storing the chunks 
vectorstore = Chroma.from_documents(
    tesla_chunks,
    embedding_model,
    collection_name=tesla_collection,
    persist_directory="./tesla_db"
)
# storing the chunks permanently
vectorstore.persist()
# Connect to the existing vector database on disk 
# This lets us skip re-reading the PDFs and re-generating embeddings
vectorstore_persisted=Chroma(
    collection_name=tesla_collection,
    persist_directory="./tesla_db",
    embedding_function=embedding_model
)


## Retreivement

In [ ]:
query = "What is the annual revenue in the year 2022 ?"

docs=vectorstore_persisted.similarity_search(query,k=5)

for i,doc in enumerate(docs):
    print(f"Retrieved chunk {i+1} : \n")
    print(doc.page_content.replace("\t"," "))
    print("\n")

## Generate  : 

In [ ]:
model_name='llama-3.3-70b-versatile'

qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer user questions only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

.
"""


qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""


In [ ]:
user_input = "What is the annual revenue in the year 2022 ?"

retriever=vectorstore_persisted.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}

)

relevant_document_chunks = retriever.invoke(user_input)

for document in relevant_document_chunks : 
    print(document.page_content.replace("/t"," "))
    break

# Composing the response
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

prompt=[
    {"role":"system","content":(qna_system_message)},
    {"role":"user","content":(qna_user_message_template.format(
        context=context_for_query,
        question=user_input
    ))}
]

try:
    response=client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )
    prediction=response.choices[0].message.content.strip()

except Exception as e :
    prediction = f"Sorry , I encountered the following error : \n {e}"

print(prediction)

In [42]:
import gradio as gr

def rag_qa(query):
    relevant_document_chunks = retriever.invoke(query)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    prompt = [
        {'role':'system', 'content': qna_system_message},
        {'role': 'user', 'content': qna_user_message_template.format(
             context=context_for_query,
             question=query
            )
        }
    ]

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=prompt,
            temperature=0
        )

        prediction = response.choices[0].message.content.strip()
    except Exception as e:
        prediction = f'Sorry, I encountered the following error: \n {e}'

    return prediction

print("Gradio imported and rag_qa function defined.")

iface = gr.Interface(fn=rag_qa, inputs='text', outputs='text', title='Tesla Annual Report QA')
iface.launch(debug=True, share=True)

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://3edeee2075cbc40cf7.gradio.live
